# VascuMap Batch Pipeline

Runs the full VascuMap pipeline on every image in a source folder.  
- **LIF files**: iterates through each image within the file  
- **TIF/TIFF files**: one pipeline run per file  
- Per-image output folder under `Z:\Bel\Vascumap_Outputs\<image_name>\`

**Default outputs** (always saved):
| File | Description |
|------|-------------|
| `*_overlay_geometry_0.tif` | 2-D device overlay with inner/outer geometry |
| `*_skeleton_overview.png` | Skeleton overview plot |
| `*_analysis_metrics.csv` | Quantitative vascular network metrics |

**Extra outputs** (when `save_all_interim = True`):
| File | Description |
|------|-------------|
| `*_cropped_stack_aligned.npy` | Brightfield stack at 2 µm iso |
| `*_vessel_translation_aligned.npy` | Pix2Pix image translation |
| `*_clean_segmentation.npy` | Cleaned binary vessel segmentation |
| `*_skeleton.npy` | Skeleton from pruned graph |
| `*_holes.npy` | Binary pore mask |
| `*_hole_labels_per_slice.npy` | Per-slice labelled pores |
| `*_hole_distance_per_slice_um.npy` | Inscribed-radius distance map |
| `*_full_graph_skeleton.npy` | Pre-pruning skeleton |
| `*_vessel_mask.npy` | Raw vessel mask |
| `*_graph_nodes.npz` | Sprout/junction node coords |
| `*_clean_graph.pkl` | NetworkX graph (pickle) |

In [1]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Point source_dir at the folder containing your .lif / .tif / .tiff files.
# All outputs land under output_base in per-image sub-folders.

source_dir  = r"Z:\Bel\Farid"    
output_base = r"Z:\Bel\Farid\Outputs"

# Set True for images whose filename contains "marina" (enables organoid masking).
# If None, the notebook auto-detects from the filename.
force_mask_central_region = None   # True / False / None (auto)

# Device width in micrometres (passed to device segmentation)
device_width_um = 35.0

# Channel index to use for multi-channel images (0-based, default 0)
channel = 0

# Set True to save extra volumes for full napari visualisation
# (holes, pore labels, full-graph skeleton, graph node coords, etc.)
save_all_interim = True

In [2]:
# ── Imports ────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

# Add bel_vascumap to the path so we can import the pipeline
sys.path.insert(0, str(Path(r"C:\Users\taylorhearn\git_repos\vascumap\bel_vascumap")))

import numpy as np
import tifffile
from liffile import LifFile

from vascumap import VascuMap
from utils import resize_dask

W0317 22:31:23.825000 619944 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ── Load models once (shared across all batch jobs) ────────────────────────────
from models import Pix2Pix, load_segmentation_model

shared_model_p2p = Pix2Pix(
    model_path=r"C:\Users\taylorhearn\git_repos\vascumap\luca_models\epoch=117-val_g_psnr=20.47-val_g_ssim=0.62.ckpt"
)
shared_model_unet = load_segmentation_model(
    r"C:\Users\taylorhearn\git_repos\vascumap\luca_models\best_full.pth"
)
print("Models loaded.")

Models loaded.


In [4]:
# ── Discover image files and build job list ────────────────────────────────────
source = Path(source_dir)
if not source.is_dir():
    raise FileNotFoundError(f"source_dir does not exist: {source}")

image_files = sorted(
    p for p in source.iterdir()
    if p.is_file() and p.suffix.lower() in (".lif", ".tif", ".tiff")
)

# Each job is (path, image_index, should_mask)
jobs = []
for p in image_files:
    should_mask = (
        force_mask_central_region
        if force_mask_central_region is not None
        else ("marina" in p.name.lower())
    )
    if p.suffix.lower() == ".lif":
        try:
            with LifFile(p) as lif:
                n_images = len(lif.images)
            for idx in range(n_images):
                img_name = lif.images[idx].name if hasattr(lif.images[idx], 'name') else ""
                if "merged" not in img_name.lower():
                    continue
                jobs.append((p, idx, should_mask))
        except Exception as exc:
            print(f"[SKIP] Could not inspect {p.name}: {exc}")
    else:
        if "merged" not in p.name.lower():
            continue
        jobs.append((p, 0, should_mask))

print(f"Source: {source}")
print(f"Files found: {len(image_files)}  |  Total jobs: {len(jobs)}")
for i, (p, idx, mask) in enumerate(jobs):
    tag = f" (LIF image {idx})" if p.suffix.lower() == ".lif" else ""
    print(f"  {i+1}. {p.name}{tag}  mask_central_region={mask}")

Source: Z:\Bel\Farid
Files found: 4  |  Total jobs: 27
  1. 03.10.25 flow 6million day 7.lif (LIF image 5)  mask_central_region=False
  2. 03.10.25 flow 6million day 7.lif (LIF image 6)  mask_central_region=False
  3. 03.10.25 flow 6million day 7.lif (LIF image 7)  mask_central_region=False
  4. 03.10.25 flow 6million day 7.lif (LIF image 8)  mask_central_region=False
  5. 03.10.25 flow 6million day 7.lif (LIF image 9)  mask_central_region=False
  6. 03.10.25 flow 8 millions day 10.lif (LIF image 5)  mask_central_region=False
  7. 03.10.25 flow 8 millions day 10.lif (LIF image 6)  mask_central_region=False
  8. 03.10.25 flow 8 millions day 10.lif (LIF image 7)  mask_central_region=False
  9. 03.10.25 flow 8 millions day 10.lif (LIF image 8)  mask_central_region=False
  10. 03.10.25 flow 8 millions day 10.lif (LIF image 9)  mask_central_region=False
  11. 03.10.25 static 6million day 4.lif (LIF image 2)  mask_central_region=False
  12. 03.10.25 static 6million day 4.lif (LIF image 3)  m

In [5]:
# ── Processing function ────────────────────────────────────────────────────────

def process_and_save(image_path: Path, image_index: int, output_base: str,
                     mask_central_region: bool = False,
                     save_all_interim: bool = False,
                     channel: int = 0,
                     model_p2p=None,
                     model_unet=None):
    """Run full VascuMap pipeline and save aligned outputs to a per-image folder."""
    vm = VascuMap(
        use_device_segmentation_app=False,
        image_source_path=str(image_path),
        image_index=image_index,
        device_width_um=device_width_um,
        mask_central_region=mask_central_region,
        channel=channel,
        model_p2p=model_p2p,
        model_unet=model_unet,
    )

    # Build a short human-readable name
    if image_path.suffix.lower() == ".lif":
        vm.image_name = f"{image_path.stem}_img{image_index}_{vm.image_name or 'image'}"
    else:
        vm.image_name = f"{image_path.stem}_{vm.image_name or 'image'}"

    output_dir = Path(output_base) / vm.image_name
    print(f"  Output → {output_dir}")

    vm.pipeline(output_dir=output_dir, save_all_interim=save_all_interim)
    return vm

In [6]:
# ── Run batch ──────────────────────────────────────────────────────────────────
MAX_RETRIES = 2
results = []          # (short_name, status, message)

for i, (image_path, image_index, mask_flag) in enumerate(jobs, start=1):
    tag = f" (LIF #{image_index})" if image_path.suffix.lower() == ".lif" else ""
    print(f"\n{'='*70}")
    print(f"[{i}/{len(jobs)}] {image_path.name}{tag}  mask_central_region={mask_flag}")
    print(f"{'='*70}")

    last_exc = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            vm = process_and_save(
                image_path=image_path,
                image_index=image_index,
                output_base=output_base,
                mask_central_region=mask_flag,
                save_all_interim=save_all_interim,
                channel=channel,
                model_p2p=shared_model_p2p,
                model_unet=shared_model_unet,
            )
            results.append((vm.image_name or image_path.stem, "OK", ""))
            last_exc = None
            break
        except Exception as exc:
            last_exc = exc
            if attempt < MAX_RETRIES:
                print(f"  ⚠ Attempt {attempt} failed: {exc}  — retrying...")
            else:
                print(f"  ✗ FAILED after {MAX_RETRIES} attempts: {exc}")

    if last_exc is not None:
        results.append((image_path.name + tag, "FAILED", str(last_exc)))

# ── Summary ───────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
n_ok = sum(1 for _, s, _ in results if s == "OK")
print(f"Batch complete: {n_ok}/{len(results)} succeeded")
if any(s == "FAILED" for _, s, _ in results):
    print("\nFailed images:")
    for name, status, msg in results:
        if status == "FAILED":
            print(f"  - {name}: {msg}")


[1/27] 03.10.25 flow 6million day 7.lif (LIF #5)  mask_central_region=False
  [LIF] Raw array shape: (2, 31, 7434, 2856)  dtype=uint8
  [LIF] 4-D → extracting channel 0 along axis 0 (shape=(2, 31, 7434, 2856))
  [LIF] Final stack shape: (31, 7434, 2856)
  [Hough fallback] Primary device detection failed (device mask touches image border) — using probabilistic Hough lines
  Output → Z:\Bel\Farid\Outputs\03.10.25 flow 6million day 7_img5_valve 1_Merged
Initial z votes {0: 0, 1: 0, 2: 0, 3: 1, 4: 0, 5: 0, 6: 1, 7: 1, 8: 0, 9: 3, 10: 1, 11: 2, 12: 3, 13: 2, 14: 9, 15: 5, 16: 8, 17: 12, 18: 10, 19: 9, 20: 7, 21: 7, 22: 2, 23: 3, 24: 4, 25: 2, 26: 0, 27: 1, 28: 6, 29: 1, 30: 0}
First cropping to z: {9: 3, 10: 1, 11: 2, 12: 3, 13: 2, 14: 9, 15: 5, 16: 8, 17: 12, 18: 10, 19: 9, 20: 7, 21: 7, 22: 2, 23: 3, 24: 4, 25: 2}
Stack width 169.9275233333333


Processing chunks: 100%|██████████| 3/3 [00:28<00:00,  9.47s/it]


strong contiguous vote planes 14-21
Running skeletonisation and analysis...


Processing chunks: 100%|██████████| 1/1 [00:04<00:00,  4.03s/it]


 chip_volume_um3  vessel_volume_um3  vessel_volume_fraction  total_vessel_length_um  vessel_length_per_chip_volume_um_inverse2  sprouts_per_vessel_length_um_inverse  junctions_per_vessel_length_um_inverse  skeleton_fractal_dimension  skeleton_lacunarity  median_sprout_and_branch_tortuosity  p90_minus_p10_sprout_and_branch_tortuosity  median_sprout_and_branch_median_cs_area_um2  p90_minus_p10_sprout_and_branch_median_cs_area_um2  median_junction_dist_nearest_junction_um  p90_minus_p10_junction_dist_nearest_junction_um  median_sprout_dist_nearest_endpoint_um  p90_minus_p10_sprout_dist_nearest_endpoint_um  total_internal_pore_count  internal_pore_area_fraction_in_filled_vascular_area  median_internal_pore_area_um2  p90_minus_p10_internal_pore_area_um2  median_internal_pore_max_inscribed_radius_um  p90_minus_p10_internal_pore_max_inscribed_radius_um  sprouts_per_chip_volume_um_inverse3  junctions_per_chip_volume_um_inverse3  total_internal_pore_density_per_vessel_volume_um_inverse3
    229

...

 chip_volume_um3  vessel_volume_um3  vessel_volume_fraction  total_vessel_length_um  vessel_length_per_chip_volume_um_inverse2  sprouts_per_vessel_length_um_inverse  junctions_per_vessel_length_um_inverse  skeleton_fractal_dimension  skeleton_lacunarity  median_sprout_and_branch_tortuosity  p90_minus_p10_sprout_and_branch_tortuosity  median_sprout_and_branch_median_cs_area_um2  p90_minus_p10_sprout_and_branch_median_cs_area_um2  median_junction_dist_nearest_junction_um  p90_minus_p10_junction_dist_nearest_junction_um  median_sprout_dist_nearest_endpoint_um  p90_minus_p10_sprout_dist_nearest_endpoint_um  total_internal_pore_count  internal_pore_area_fraction_in_filled_vascular_area  median_internal_pore_area_um2  p90_minus_p10_internal_pore_area_um2  median_internal_pore_max_inscribed_radius_um  p90_minus_p10_internal_pore_max_inscribed_radius_um  sprouts_per_chip_volume_um_inverse3  junctions_per_chip_volume_um_inverse3  total_internal_pore_density_per_vessel_volume_um_inverse3
    358

Processing chunks: 100%|██████████| 4/4 [00:49<00:00, 12.30s/it]


strong contiguous vote planes 24-33
Running skeletonisation and analysis...


Processing chunks: 100%|██████████| 1/1 [00:05<00:00,  5.43s/it]


 chip_volume_um3  vessel_volume_um3  vessel_volume_fraction  total_vessel_length_um  vessel_length_per_chip_volume_um_inverse2  sprouts_per_vessel_length_um_inverse  junctions_per_vessel_length_um_inverse  skeleton_fractal_dimension  skeleton_lacunarity  median_sprout_and_branch_tortuosity  p90_minus_p10_sprout_and_branch_tortuosity  median_sprout_and_branch_median_cs_area_um2  p90_minus_p10_sprout_and_branch_median_cs_area_um2  median_junction_dist_nearest_junction_um  p90_minus_p10_junction_dist_nearest_junction_um  median_sprout_dist_nearest_endpoint_um  p90_minus_p10_sprout_dist_nearest_endpoint_um  total_internal_pore_count  internal_pore_area_fraction_in_filled_vascular_area  median_internal_pore_area_um2  p90_minus_p10_internal_pore_area_um2  median_internal_pore_max_inscribed_radius_um  p90_minus_p10_internal_pore_max_inscribed_radius_um  sprouts_per_chip_volume_um_inverse3  junctions_per_chip_volume_um_inverse3  total_internal_pore_density_per_vessel_volume_um_inverse3
    343

In [7]:
# ── Maria batch: discover jobs (ALL images, no name filter) ────────────────────
maria_source_dir  = r"Z:\Bel\Maria"
maria_output_base = r"Z:\Bel\Maria\Outputs"

maria_source = Path(maria_source_dir)
if not maria_source.is_dir():
    raise FileNotFoundError(f"source_dir does not exist: {maria_source}")

maria_image_files = sorted(
    p for p in maria_source.iterdir()
    if p.is_file() and p.suffix.lower() in (".lif", ".tif", ".tiff")
)

maria_jobs = []
for p in maria_image_files:
    if p.suffix.lower() == ".lif":
        try:
            with LifFile(p) as lif:
                n_images = len(lif.images)
            for idx in range(n_images):
                maria_jobs.append((p, idx, False))
        except Exception as exc:
            print(f"[SKIP] Could not inspect {p.name}: {exc}")
    else:
        maria_jobs.append((p, 0, False))

print(f"Source: {maria_source}")
print(f"Files found: {len(maria_image_files)}  |  Total jobs: {len(maria_jobs)}")
for i, (p, idx, mask) in enumerate(maria_jobs):
    tag = f" (LIF image {idx})" if p.suffix.lower() == ".lif" else ""
    print(f"  {i+1}. {p.name}{tag}")

Source: Z:\Bel\Maria
Files found: 4  |  Total jobs: 90
  1. exp6_260217_Vascumap.lif (LIF image 0)
  2. exp6_260217_Vascumap.lif (LIF image 1)
  3. exp6_260217_Vascumap.lif (LIF image 2)
  4. exp6_260217_Vascumap.lif (LIF image 3)
  5. exp6_260217_Vascumap.lif (LIF image 4)
  6. exp6_260217_Vascumap.lif (LIF image 5)
  7. exp6_260217_Vascumap.lif (LIF image 6)
  8. exp6_260217_Vascumap.lif (LIF image 7)
  9. exp6_260217_Vascumap.lif (LIF image 8)
  10. exp6_260217_Vascumap.lif (LIF image 9)
  11. exp6_260217_Vascumap.lif (LIF image 10)
  12. exp6_260217_Vascumap.lif (LIF image 11)
  13. exp6_260217_Vascumap.lif (LIF image 12)
  14. exp6_260217_Vascumap.lif (LIF image 13)
  15. exp6_260217_Vascumap.lif (LIF image 14)
  16. exp6_260217_Vascumap.lif (LIF image 15)
  17. exp6_260217_Vascumap.lif (LIF image 16)
  18. exp6_260217_Vascumap.lif (LIF image 17)
  19. exp6_260217_Vascumap.lif (LIF image 18)
  20. exp6_260217_Vascumap.lif (LIF image 19)
  21. exp6_260217_Vascumap.lif (LIF image 20

In [ ]:
# ── Maria batch: run ───────────────────────────────────────────────────────────
MAX_RETRIES = 2
maria_results = []

for i, (image_path, image_index, mask_flag) in enumerate(maria_jobs, start=1):
    tag = f" (LIF #{image_index})" if image_path.suffix.lower() == ".lif" else ""
    print(f"\n{'='*70}")
    print(f"[{i}/{len(maria_jobs)}] {image_path.name}{tag}")
    print(f"{'='*70}")

    last_exc = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            vm = process_and_save(
                image_path=image_path,
                image_index=image_index,
                output_base=maria_output_base,
                mask_central_region=False,
                save_all_interim=False,
                channel=channel,
                model_p2p=shared_model_p2p,
                model_unet=shared_model_unet,
            )
            maria_results.append((vm.image_name or image_path.stem, "OK", ""))
            last_exc = None
            break
        except Exception as exc:
            last_exc = exc
            if attempt < MAX_RETRIES:
                print(f"  ⚠ Attempt {attempt} failed: {exc}  — retrying...")
            else:
                print(f"  ✗ FAILED after {MAX_RETRIES} attempts: {exc}")

    if last_exc is not None:
        maria_results.append((image_path.name + tag, "FAILED", str(last_exc)))

# ── Summary ───────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
n_ok = sum(1 for _, s, _ in maria_results if s == "OK")
print(f"Maria batch complete: {n_ok}/{len(maria_results)} succeeded")
if any(s == "FAILED" for _, s, _ in maria_results):
    print("\nFailed images:")
    for name, status, msg in maria_results:
        if status == "FAILED":
            print(f"  - {name}: {msg}")


[1/90] exp6_260217_Vascumap.lif (LIF #0)
  [LIF] Raw array shape: (21, 6380, 3861)  dtype=uint8
  [LIF] Final stack shape: (21, 6380, 3861)
  [Hough fallback] Primary device detection failed (device mask touches image border) — using probabilistic Hough lines
  Output → Z:\Bel\Maria\Outputs\exp6_260217_Vascumap_img0_A3-1 Paclitaxel
Initial z votes {0: 1, 1: 0, 2: 0, 3: 4, 4: 3, 5: 6, 6: 4, 7: 14, 8: 9, 9: 8, 10: 10, 11: 3, 12: 4, 13: 2, 14: 4, 15: 0, 16: 6, 17: 5, 18: 2, 19: 4, 20: 11}
First cropping to z: {0: 1, 1: 0, 2: 0, 3: 4, 4: 3, 5: 6, 6: 4, 7: 14, 8: 9, 9: 8, 10: 10, 11: 3, 12: 4, 13: 2, 14: 4, 15: 0, 16: 6}
Stack width 169.99218


Processing chunks: 100%|██████████| 3/3 [00:09<00:00,  3.29s/it]


strong contiguous vote planes 3-12
Running skeletonisation and analysis...


Processing chunks: 100%|██████████| 1/1 [00:01<00:00,  1.24s/it]


 chip_volume_um3  vessel_volume_um3  vessel_volume_fraction  total_vessel_length_um  vessel_length_per_chip_volume_um_inverse2  sprouts_per_vessel_length_um_inverse  junctions_per_vessel_length_um_inverse  skeleton_fractal_dimension  skeleton_lacunarity  median_sprout_and_branch_tortuosity  p90_minus_p10_sprout_and_branch_tortuosity  median_sprout_and_branch_median_cs_area_um2  p90_minus_p10_sprout_and_branch_median_cs_area_um2  median_junction_dist_nearest_junction_um  p90_minus_p10_junction_dist_nearest_junction_um  median_sprout_dist_nearest_endpoint_um  p90_minus_p10_sprout_dist_nearest_endpoint_um  total_internal_pore_count  internal_pore_area_fraction_in_filled_vascular_area  median_internal_pore_area_um2  p90_minus_p10_internal_pore_area_um2  median_internal_pore_max_inscribed_radius_um  p90_minus_p10_internal_pore_max_inscribed_radius_um  sprouts_per_chip_volume_um_inverse3  junctions_per_chip_volume_um_inverse3  total_internal_pore_density_per_vessel_volume_um_inverse3
     93

...

Processing chunks: 100%|██████████| 3/3 [00:10<00:00,  3.41s/it]


strong contiguous vote planes 4-14
Running skeletonisation and analysis...


Processing chunks: 100%|██████████| 1/1 [00:01<00:00,  1.20s/it]


 chip_volume_um3  vessel_volume_um3  vessel_volume_fraction  total_vessel_length_um  vessel_length_per_chip_volume_um_inverse2  sprouts_per_vessel_length_um_inverse  junctions_per_vessel_length_um_inverse  skeleton_fractal_dimension  skeleton_lacunarity  median_sprout_and_branch_tortuosity  p90_minus_p10_sprout_and_branch_tortuosity  median_sprout_and_branch_median_cs_area_um2  p90_minus_p10_sprout_and_branch_median_cs_area_um2  median_junction_dist_nearest_junction_um  p90_minus_p10_junction_dist_nearest_junction_um  median_sprout_dist_nearest_endpoint_um  p90_minus_p10_sprout_dist_nearest_endpoint_um  total_internal_pore_count  internal_pore_area_fraction_in_filled_vascular_area  median_internal_pore_area_um2  p90_minus_p10_internal_pore_area_um2  median_internal_pore_max_inscribed_radius_um  p90_minus_p10_internal_pore_max_inscribed_radius_um  sprouts_per_chip_volume_um_inverse3  junctions_per_chip_volume_um_inverse3  total_internal_pore_density_per_vessel_volume_um_inverse3
    103

 25%|██▌       | 583/2328 [01:27<04:20,  6.70it/s]